# Model Experiments - Telco Customer Churn

En este notebook se prueban diferentes modelos de machine learning
para predecir el abandono de clientes (churn).

Objetivo:
- Comparar modelos
- Evaluar métricas
- Identificar el mejor enfoque.

Dataset utilizado:
Telco Customer Churn Dataset

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score


## Cargar dataset limpio

En este notebook se utiliza el dataset procesado generado por el pipeline.
Flujo del proyecto:

raw → limpieza → processed → modelos

In [2]:
df = pd.read_csv("../data/processed/telco_churn_clean.csv")
df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Separación de variables


In [3]:
X = df.drop("Churn", axis=1)
y = df["Churn"]


## División en entrenamiento y prueba


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


## Preprocesamiento de datos


In [5]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)


## Modelo 1: Logistic Regression

In [6]:
log_model = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

log_model.fit(X_train, y_train)
preds = log_model.predict(X_test)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

          No       0.85      0.90      0.87      1031
         Yes       0.67      0.54      0.60       371

    accuracy                           0.81      1402
   macro avg       0.76      0.72      0.74      1402
weighted avg       0.80      0.81      0.80      1402



## Modelo 2: Random Forest

In [7]:
rf_model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

rf_model.fit(X_train, y_train)
preds = rf_model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

          No       0.83      0.90      0.86      1031
         Yes       0.64      0.48      0.55       371

    accuracy                           0.79      1402
   macro avg       0.73      0.69      0.70      1402
weighted avg       0.78      0.79      0.78      1402



## Comparación de modelos

In [9]:
models = {
    "Logistic": log_model,
    "RandomForest": rf_model
}

for name, model in models.items():
    preds = model.predict(X_test)
    score = f1_score(y_test, preds, pos_label="Yes")
    print(name, score)


Logistic 0.5991058122205664
RandomForest 0.5454545454545454


## Conclusiones

- Logistic Regression mostró buen equilibrio entre precisión y recall.
- Random Forest capturó relaciones no lineales.

Próximos pasos:
- ajuste de hiperparámetros
- selección de variables
- optimización del threshold
